# Gold Customers

**Autor:** Daniel Guzmán  
**Actividad:** Actividad 04 — Arquitectura Medallón  
**Capa:** Gold  
**Entorno:** Databricks


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

df_gold = spark.read.table("workspace.default.silver_customers")

print(f"Registros en Silver: {df_gold.count()}")
df_gold.printSchema()
display(df_gold.limit(5))

In [0]:
df_gold.createOrReplaceTempView("silver_customers_view")

In [0]:
df_gold_result = spark.sql("""
    SELECT
        country,
        COUNT(*) AS total_customers,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
    FROM silver_customers_view
    GROUP BY country
    ORDER BY total_customers DESC
""")

display(df_gold_result)

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_customers_by_country")

df_gold_result.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customers_by_country")

In [0]:
df_check = spark.read.table("workspace.default.gold_customers_by_country")

print(f"Registros en gold_customers_by_country: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(10))

In [0]:
bronze_count = spark.read.table("workspace.default.bronze_customers").count()
silver_count = spark.read.table("workspace.default.silver_customers").count()
gold_count = spark.read.table("workspace.default.gold_customers_by_country").count()

print(f"Bronze: {bronze_count}")
print(f"Silver: {silver_count}")
print(f"Gold: {gold_count}")
print(f"Registros eliminados entre Bronze y Silver: {bronze_count - silver_count}")

In [0]:
bronze_count = spark.read.table("workspace.default.bronze_customers").count()
silver_count = spark.read.table("workspace.default.silver_customers").count()
gold_count = spark.read.table("workspace.default.gold_customers_by_country").count()

print(f"Bronze: {bronze_count}")
print(f"Silver: {silver_count}")
print(f"Gold: {gold_count}")